In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
import time
from sklearn.metrics import classification_report, accuracy_score, f1_score
import os
from imblearn.over_sampling import SMOTE

In [2]:
# Constants
DATA_PATH = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
MODEL_NAME = "StackedEnsemble_RF (SMOTE)"
DATASET_NAME = "GPS_Data_Simplified_2D_Feature_Map.xlsx"
RESULTS_CSV = "../../../2D_Results.csv"
RANDOM_STATE = 1082

In [3]:
# Load the dataset
try:
    df = pd.read_excel(DATA_PATH)
except FileNotFoundError:
    print(f"Error: The file '{DATA_PATH}' was not found.")
    print("Please make sure the path is correct.")
    exit()

In [4]:
# Prepare features and labels
X = df.drop('Output', axis=1)
y = df['Output']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.8, random_state=RANDOM_STATE, stratify=y
)

# Unsupervised stages see only the benign rows of the training split
X_fit_raw = X_train_raw[y_train == 0]

scaler = StandardScaler()
scaler.fit(X_fit_raw)
X_fit_scaled = scaler.transform(X_fit_raw)
X_train_scaled = scaler.transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# PCA for OC-SVM and LOF (85% variance)
pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
pca_svm_lof.fit(X_fit_scaled)
X_fit_pca_svm_lof = pca_svm_lof.transform(X_fit_scaled)
X_train_pca_svm_lof = pca_svm_lof.transform(X_train_scaled)
X_test_pca_svm_lof = pca_svm_lof.transform(X_test_scaled)

# PCA for Autoencoder (95% variance)
pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
pca_ae.fit(X_fit_scaled)
X_fit_pca_ae = pca_ae.transform(X_fit_scaled)
X_train_pca_ae = pca_ae.transform(X_train_scaled)
X_test_pca_ae = pca_ae.transform(X_test_scaled)

In [5]:
# Train base models
start_time_total = time.time()

# One-Class SVM
ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
ocsvm.fit(X_fit_pca_svm_lof)

# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
lof.fit(X_fit_pca_svm_lof)

# Autoencoder using MLP
ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=RANDOM_STATE)
ae.fit(X_fit_pca_ae, X_fit_pca_ae)


def stack_scores(X_pca_svm_lof, X_pca_ae):
    recon = ae.predict(X_pca_ae)
    ae_score = np.mean((X_pca_ae - recon) ** 2, axis=1)
    return np.vstack([
        ocsvm.decision_function(X_pca_svm_lof),
        lof.decision_function(X_pca_svm_lof),
        ae_score,
    ]).T


X_meta_train = stack_scores(X_train_pca_svm_lof, X_train_pca_ae)
base_train_time = time.time() - start_time_total

X_meta_test = stack_scores(X_test_pca_svm_lof, X_test_pca_ae)
y_meta_train = y_train
y_meta_test = y_test

In [6]:
# meta classifier
start_meta = time.time()

# Calculate the count of the majority class for SMOTE
count_of_majority_class = y_meta_train.value_counts().max()
desired_strategy = {
    1: count_of_majority_class,
    2: count_of_majority_class,
    3: count_of_majority_class
}

# Apply SMOTE to the meta-classifier training data
smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=desired_strategy)
X_meta_train_bal, y_meta_train_bal = smote.fit_resample(X_meta_train, y_meta_train)

# Meta-classifier (Random Forest) for multiclass classification
meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
meta_clf.fit(X_meta_train_bal, y_meta_train_bal)
total_train_time = base_train_time + (time.time() - start_meta)

stacked_pred = meta_clf.predict(X_meta_test)


def predict_end_to_end(rows):
    rows_scaled = scaler.transform(rows)
    return meta_clf.predict(stack_scores(pca_svm_lof.transform(rows_scaled),
                                         pca_ae.transform(rows_scaled)))

In [7]:
# Evaluation
report = classification_report(
    y_meta_test,
    stacked_pred,
    labels=[0, 1, 2, 3],
    target_names=["benign", "attack1", "attack2", "attack3"],
    zero_division=0,
    output_dict=True
)
f1 = report["weighted avg"]["f1-score"]
accuracy = accuracy_score(y_meta_test, stacked_pred)
print(classification_report(
    y_meta_test,
    stacked_pred,
    labels=[0, 1, 2, 3],
    target_names=["benign", "attack1", "attack2", "attack3"],
    zero_division=0
))
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {accuracy:.4f}")

              precision    recall  f1-score   support

      benign       1.00      0.99      0.99    397825
     attack1       0.94      0.99      0.97     36458
     attack2       0.99      1.00      1.00     44232
     attack3       0.95      0.99      0.97     32015

    accuracy                           0.99    510530
   macro avg       0.97      0.99      0.98    510530
weighted avg       0.99      0.99      0.99    510530

F1 Score: 0.9909
Accuracy: 0.9908


In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = meta_clf.predict_proba(X_meta_test) if hasattr(meta_clf, "predict_proba") else None
_timing = time_inference(predict_end_to_end, X_test_raw[:2000])

row = evaluate(
    y_meta_test,
    stacked_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="SMOTE",
    labels=[0, 1, 2, 3],
    class_names=["benign", "simple", "intermediate", "sophisticated"],
    y_proba=_proba,
    y_train=y_meta_train_bal,
    train_time_sec=total_train_time,
    num_epochs="N/A",
    parameters=meta_clf.get_params(),
    seed=RANDOM_STATE,
    n_features=X_test_raw.shape[1],
    test_set="real_heldout_20pct",
    n_train=len(y_meta_train_bal),
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")